# Processing Metabolomic NMR Data – A Practical Guide
*Marcel Utz, KIT, 2026*

## Contents
1. Basics and Mathematical Foundation
2. Processing Spectra with `NMRlab`
3. Quantitative Information from Series of Spectra

## 1 Basics and Mathematical Foundation

The goal of metabolomic NMR data analysis is to determine the composition of the mixture under study.
That sounds simple enough, but requires us to contemplate what we do and typically do not know
about our sample first.

Metabolic mixtures are solutions, most often in an aqueous medium, of a large variety of
organic molecules which are either the products or the educts of physiological processes.
They typically contain
- a solvent (often water)
- inorganic ions
- small organic molecules
- macromolecules (proteins, polysaccharides)
- lipid aggregates (micelles and liposomes)
- cells (live and dead)

What we typically are after are the *concentrations* of a subset of the organic
molecules present. 

However, one may analyse metabolic mixtures with a different agenda. For example,
the aim may be to discover novel or unexpected molecules, which may be present at
low concentration, or substances that are somewhat intermediate between macromolecules
and small molecules, such as peptides, nucleic acids, or saccharides of various lengths
and topologies.

In an abstract mathematical sense, a complex mixture is characterised of
a *solvent* $s_0$ in combination with a list of dissolved species $s_k$, $1\le k \le n$.
A complete description of the mixture can be given by a vector of amounts $n_k>0$, $0\le k$.
The aim of metabolic analysis is to determine some or all of these amounts from an experimental
spectrum.

A *spectrum* is the result of an experiment that exposes the complex mixture to some
kind of radiation, and produces a function $S(\omega)$, where $\omega$ is the spectral
variable. In the case of NMR, $\omega$ is usually a frequency or a chemical shift value. 
For other techniques, $\omega$ may represent a different physical quantity, such as the $m/Z$ value
in the case of mass spectrometry, or a wavelength or wavenumber in the case of optical or
IR spectroscopy.

The basic idea is that the spectrum depends on the amounts in a known manner, such
that the amounts can be determined once the spectrum has been measured. Mathematically,
we can regard the measured spectrum as a function not only of $\omega$, but of the amounts
$n_k$, as well:
$$ S(\omega) = S(\{n_k\};\omega). $$
In reality, the spectrum we observe is *not* fully determined by the amounts. We need to
allow for experimental noise, which adds a stochastic component to the spectrum:
$$ S_\text{obs}(\omega) = S(\{n_k\};\omega)+ \tilde N(\omega),$$
where the tilde signals that $N(\omega)$ is a stochastic variable.

The problem of analysing a complex mixture can therefore be formulated as follows:

> Determine the amounts ${n_k}$ such that the observed spectrum is represented
> as closely as possible by the function $S({n_k};\omega)$. 

Of course, in practice, this presumes that we have a way of calculating the spectrum
for different combinations of $\{n_k\}$. Even if we do, finding the right combination
of $n_k$ is a typical inverse problem. Such problems are notoriously hard to solve in
general - and that is assuming that we know in
advance which compounds $s_k$ to expect in the mixture. In most practical cases, there
may be compounds present that we do not know. Solving the problem above is therefore
notoriously *difficult*.

### Partial Spectra and the Gibbs-Duhem Equation

The *partial spectrum* $S_k(\{n_k\},\omega)$ is defined as
$$
S_k(\{n_k\},\omega) = \left(\frac{\partial S({n_k};\omega) }{\partial n_k}\right)_{n_{j\ne k};\omega}. 
$$
In NMR, we can assume that the spectrum is *homogeneous* of degree 1 in the amounts $n_k$. This basically means
that the doubling any of the amounts will increase the contribution of the compound in question to the
spectrum proportionally. Formally, we have then
$$ 
S_k(\lambda \{n_k\},\omega) = \lambda S_k(\{n_k\};\omega).
$$ 
It can be shown by using Euler's homogeneous function theorem that in this case, the spectrum is an additive superposition of the 
partial spectra:
$$
S_k(\{n_k\},\omega) = \sum n_k\, S_k(\{n_k\};\omega).
$$
This is, by the way, analogous to the famous Gibbs-Duhem equation of Thermodynamics, which
relates the amounts and chemical potentials of compounds in mutual thermodynamic equilibrium.
If, on top of this, we can assume that the partial spectra for all compounds are indendent of
the amounts present, we arrive at the simple additive formula
$$
S(\{n_k\},\omega) = \sum n_k\, S_k(\omega).
$$
In this case, the spectrum is a very simple additive superposition of mutually independent 
contributions from each compound present. To a good approximation, this is the 
case for NMR spectra in the context of metabolomics.

> NOTE: 
> We have deliberately used the amounts and *not* the concentrations. The argument is based on 
> scaling of the *entire* sample, including the solvent. This means that the scaling
> conserves the concentrations of all species in the sample, and therefore does 
> not affect things such as pH etc.






## 2 Processing NMR Data

We use the `NMRlab.jl` package to process a sequence of NMR spectra. The package is under development by Manaz, but is very close
to its first public release. Documentation can be found at [https://abragam.imt.kit.edu/group/nmrlab-docs/](https://abragam.imt.kit.edu/group/nmrlab-docs/index.html).

> NOTE: "NMRlab" is a working name. For the public release, we plan to change the name, since the original one is already taken by an unrelated project.

At the same time, we use the `Plots`package for creating plots, and we set a few defaults for their appearance. These values are optimised for
plots that go into publications. You may of course change these as you see fit. If you generate plots for presentations, you may want to increase the font
size for the ticks and legend further.

In [ ]:
import Pkg
Pkg.activate(".")
Pkg.resolve()
# Pkg.add("Revise")
#Pkg.instantiate()
using Revise
#Pkg.rm("NMRlab")
#Pkg.develop(path="/Users/mu3q/Dropbox/Source/NMRlab.jl")
#Pkg.add(url="https://github.com/marcel-utz/NMRlab.jl")

using NMRlab, Plots
theme(:dark)
default(size=(600,400),
        tick_direction=:out,
        framestyle=:box,
        margin=(6,:mm),
        legendfontsize=8,
        guidefontsize=10,
        tickfontsize=10,
        fontfamily="Helvetica", 
        fmt=:svg,dpi=100)

As explained in the documentation, `NMRlab` comes with some example data sets. Here is a plot of a Bruker FID:

In [ ]:
using NMRlab
using NMRlab.Examples
using Plots: plot, savefig

data_bruker = NMRlab.Examples.Data["HCC cell culture media spectra"]

# High level vendor loader (recommended)
params_bruker, data_td = NMRlab.load(joinpath(data_bruker["path"], "10"), :Bruker)

t = data_td.coord[1]
y = real.(data_td.dat)

plot(t, y;
xlabel = "time / s",
ylabel = "signal (a.u.)",
title = "Bruker FID (real part)")

### Accessing Demonstration Data
Yaping has kindly provided a set of demonstration data files for us to use in this demo. They consist of three repeats of a 1D NMR experiment on  the supernatant of a cell culture
    taken at different time points. The data is stored in JEOL format.
Julia offers convenient tools to traverse the file system:

In [ ]:
for (path, dirs, files) in walkdir("./NMR data --monolayer")
    print("Directory: ", path, "\n")
    for file in files
        print("  File: ", joinpath(path,file), "\n")
    end
end 

We can use this to read all data and assemble it into a `SpectData` object.

In [ ]:
fids=[]
fnames=[]

for (path, dirs, files) in walkdir("./NMR data --monolayer")
    print("Directory: ", path, "\n")
    for file in files
        print("  File: ", joinpath(path,file), "\n")
        params,data=NMRlab.load(joinpath(path,file), :JEOL)
        push!(fids, data) 
        push!(fnames, basename(file))
    end
end 

fidData=SpectData(hcat(fids...),(NMRlab.coords(fids[1],1) ,fnames))  # combine all FIDs into a single data object

We now have a two-dimenisonal `SpectData` object with the time as the first axis and the file name as the second:

In [ ]:
NMRlab.coords(fidData,2)  # check that the file names are stored as 2nd dimension coordinates

We need to process this raw time-domain data to obtain spectra. For this, we define
a processing stack:

In [ ]:
process = Chain(
    ZeroFill([2^16,:]),
    Apodize([0.5pi,:]),
    FourierTransform([2^16,18],[1]),
    x-> x[10000:end-10000,:],  # crop the spectrum to the region of interest
    PhaseCorrect(0.4pi,2pi*0.00172,1),
    NMRlab.AutoPhaseCorrectChen(1,verbose=false,γ=0.0e-5),
    # MedianBaselineCorrect(1, wdw=1000)
)

processedData = process(fidData)  # apply the processing chain to the FIDs

n=1
plot(NMRlab.coords(processedData,1)./600 .+ 4.78, real.(processedData[:,n]);
xlabel = "chemical shift (ppm)",
ylabel = "signal (a.u.)",
xaxis = :flip,
title = NMRlab.coords(processedData,2)[n], xlims=(-0.1,9.0),ylims=1e4 .* (-0.1,1.05)) 

In [ ]:
# processedData[32500:33500,:] .= 0.0
a=plot()  # create an empty plot object to hold all spectra
for n in 1:size(processedData,2)    
    plot!(a,NMRlab.coords(processedData,1)./600 .+ 4.78, 200n .+ real.(processedData[:,n]);
        xlabel = "chemical shift (ppm)",
        ylabel = "signal (a.u.)",
        xaxis = :flip,
        ylims=5e3 .* (-0.1,1.05),
        legend=false)
end
savefig(a, "processed_spectra.svg")
a

In [ ]:
ELandScape=[ processedData[:,9] |> PhaseCorrect(x1,x2,1) |> Derivative(1) |> NMRlab.entropy
     for x1 in range(-pi/2,pi/2,length=40), 
         x2 in range(-0.005,0.005,length=40) ]
Plots.contourf(range(-pi/2,pi/2,length=40), range(-0.005,0.005,length=40), ELandScape';
     xlabel="zero order phase / rad", 
     ylabel="first order phase / rad s", 
     title="entropy landscape", 
     colorbar=false, 
     size=(600,600),)   

In [ ]:
processedData = processedData |> MedianBaselineCorrect(1, wdw=10000) # apply baseline correction to the processed data

In [ ]:
a=plot()  # create an empty plot object to hold all spectra
for n in 1:size(processedData,2)    
    plot!(a,NMRlab.coords(processedData,1)./600 .+ 4.78, 5000*(n-1) .+ processedData[:,n] ;
        xlabel = "chemical shift (ppm)",
        size=(600,800),
        xaxis = :flip,
        yticks=false,
        ylims=5e4 .* (-0.1,2.05),
        annotationfontsize=8,
        annotations=(10.0, 5000*(n-1)+1000,NMRlab.coords(processedData,2)[n]),
        legend=false)
end
a

In [ ]:
using JLD2
@save "processedData.jld2" processedData fidData

## Deconvolution of Experimental Spectra

We can now proceed to determining the amounts in the experimental spectra. 
The partial spectra have been computed separately, using the same grid as the experimental ones. We load them
from a `.jld2` file:

In [ ]:
@load "partial_spectra.jld2"

### Alignment

The peak positions in the partial spectra must correspond to the ones in the experimental data, otherwise the determination of
amounts cannot be accurate. We can check this by examining the positions of certain prominent peaks that are easy to assign
in the experimental spectra. 

For the TSP peak, the alignment seems quite good, but not perfect in all cases of the experimental spectra:

In [ ]:
Plots.plot(NMRlab.coords(partialSpects,1)./600 .+ 4.78, 5*real(partialSpectra["TSP"]), xaxis=:flip, 
    label=NMRlab.coords(partialSpects,2)[2],
    legend=:none)
Plots.plot!(NMRlab.coords(processedData,1)./600 .+ 4.78, 2e3 .+ real.(processedData[:,:] ), xaxis=:flip, 
label="processed",xlims=0.0.+(-0.1,0.1),ylims=5e3 .* (-0.1,1.5))

We correct this by using the processor `PeakAlign`:

In [ ]:
processedData = processedData |> PeakAlign(1,-4.7995*600,100) ;  # align the spectra to the TSP peak at 0 ppm

In [ ]:
Plots.plot(NMRlab.coords(partialSpects,1)./600 .+ 4.78, 5*real(partialSpectra["TSP"]), xaxis=:flip, 
    label=NMRlab.coords(partialSpects,2)[2],
    legend=false)
Plots.plot!(NMRlab.coords(processedData,1)./600 .+ 4.78, 2e3 .+ real.(processedData[:,:] ), xaxis=:flip, 
label="processed",xlims=0.0.+(-0.1,0.1),ylims=5e3 .* (-0.1,1.5))

Unfortunately, this is not the entire story. We have to verify that the metabolites appear at  the chemical shift values contained in the GISSMO database. We cannot check this here for all peaks and all metabolites, but we do so for the 
anomeric peak of $\alpha$-Glucose as an example:

In [ ]:
Plots.plot(NMRlab.coords(partialSpects,1)./600 .+ 4.78, 5*real(partialSpectra["(+/-)-Glucose"]), xaxis=:flip, 
    label=NMRlab.coords(partialSpects,2)[2],
    legend=false)
Plots.plot!(NMRlab.coords(processedData,1)./600 .+ 4.78, 2e3 .+ real.(processedData[:,:] ), xaxis=:flip, 
label="processed",xlims=0.0.+(5.1,5.3),ylims=5e3 .* (-0.1,1.5))

At this point, it makes sense to save the processed spectra in a JLD2 file so they can be recalled later:

In [ ]:
@save "processed_spectra.jld2" processedData

### Spectral Fits

We would like to represent the experimental spectra as a superposition of partial spectra. In the most straightforward approach, we solve the additive formula
$$
S(\{n_k\},\omega) = \sum n_k\, S_k(\omega).
$$
for the vector of amounts $(n_k)$.

In the language of linear algebra, we can write this as
$$
\mathbf{P} \mathbf{n} = \mathbf{s},
$$
where $\mathbf{P}$ is the matrix of $l$ partial spectra, consisting of $m$ data points each, $\mathbf{s}$ is the experimental spectrum (of $m$ points), and $\mathbf{n}$ represents the vector of amounts.

 However, we cannot hope for
a perfect fit, since the experimential spectrum contains
noise and other artefacts, as well as compounds for which we
do not know the partial spectra. This is also reflected
in the fact that $\mathbf{P}$ is not square and therefore singular. The best we can
do is to minimize the residual by a least squares approach.

To this effect, we compute the pseudoinverse of the matrix of partial spectra:

In [ ]:
import LinearAlgebra
B = SpectData(partialSpects |> real |> LinearAlgebra.pinv,(NMRlab.coords(partialSpects,2), NMRlab.coords(partialSpects,1)))

This can then simply be multiplied with the matrix of experimental spectra to obtain the raw amounts:

In [ ]:
rawamounts=SpectData(B  * processedData , (NMRlab.coords(B,1), NMRlab.coords(processedData,2)))

In [ ]:
Plots.heatmap(rawamounts,
    xticks=(1:18,NMRlab.coords(rawamounts,2)),xrotation=60,
    yticks=(1:38,NMRlab.coords(rawamounts,1)),
    size=(800,800))

In [ ]:
Plots.bar(
     NMRlab.coords(rawamounts,1), 
     rawamounts[:,2], 
     xticks=(0.5:38, NMRlab.coords(rawamounts,1)),
     legend=false,
     size=(1000,600),
     ylabel="relative amount (a.u.)", title=NMRlab.coords(rawamounts,2)[1],xrotation=60,)

In [ ]:
n=3
fittedSpectra = SpectData(partialSpects * rawamounts, (NMRlab.coords(partialSpects,1), NMRlab.coords(rawamounts,2)))
Plots.plot(NMRlab.coords(fittedSpectra,1)./600 .+ 4.78, real(fittedSpectra)[:,n], xaxis=:flip, fmt=:svg,label="fitted spectrum",legend=:topleft)
Plots.plot!(NMRlab.coords(processedData,1)./600 .+ 4.78, 0.5e3 .+ real.(processedData[:,n]), xaxis=:flip, fmt=:svg, 
label="measured spectrum",ylims=10e3 .* (-0.1,1.5),xlims=0.0.+(-0.1,8.7))

The fit is reasonable, but has a few quirks. Immediately obvious are negative contributions. The linear approach taken happily uses negative amounts, even though
they are not physical. It also tends to produce noisy amount vectors if there are 
partial spectra of different compounds with large peak overlap: the solution found
may contain two components that have similar peak positions, one with a positive and
one with a negative amount, thus largely compensating each other.

### Tikhonov Regularisation

There are several numerical approaches to deal with this. The first one is Tikhonov regularisation, which can be used to bias the solution towards a certain quality measure, while sacrificing some fit accuracy. In our case, we may prefer a "simple" solution, containing only a small number of compounds, over a "complicated" one. 
The bias can be controlled through the variable $\lambda$. A good value for it has
to be found by trial and error; too small a value has no effect, whereas too large a value will give a zero solution.

In [ ]:
λ = 1e7
B_tikonov = SpectData(partialSpects |> real |> x -> (x'*x + λ*LinearAlgebra.I)\x',(NMRlab.coords(partialSpects,2), NMRlab.coords(partialSpects,1)))  

In [ ]:
rawamounts_tikonov=SpectData(B_tikonov  * processedData , (NMRlab.coords(B_tikonov,1), NMRlab.coords(processedData,2)))

In [ ]:
Plots.bar(
     NMRlab.coords(rawamounts_tikonov,1), 
     rawamounts_tikonov[:,2], 
     xticks=(0.5:39, NMRlab.coords(rawamounts_tikonov,1)),
     legend=false,
     size=(900,600),
     ylabel="relative amount (a.u.)", title=NMRlab.coords(rawamounts,2)[1],xrotation=60,)

In [ ]:
n=3
fittedSpectra = SpectData(partialSpects * rawamounts_tikonov, (NMRlab.coords(partialSpects,1), NMRlab.coords(rawamounts_tikonov,2)))
Plots.plot(NMRlab.coords(fittedSpectra,1)./600 .+ 4.78, real(fittedSpectra)[:,n], xaxis=:flip, fmt=:svg,label="fitted spectrum",legend=:topleft)
Plots.plot!(NMRlab.coords(processedData,1)./600 .+ 4.78, 0.5e3 .+ real.(processedData[:,n]), xaxis=:flip, fmt=:svg, 
label="measured spectrum",ylims=10e3 .* (-0.1,1.5),xlims=0.0.+(-0.1,8.7))

### Non-Negative Least Squares

While Tikhonov regularisation biases the solution towards low amounts, NNLS provides a way to minimize the least squares residual subject to the constraint
of non-negative amounts. We use a Julia package to access this method. Unlike before, we have to process each spectrum in turn, 
we can no longer compute a $B$ matrix which could be multiplied with the matrix of experimental spectra. Here is the necessary code:

In [ ]:
import NonNegLeastSquares

n_samples = size(processedData,2)
n_met = size(partialSpects,2)

rawamounts_nnls = SpectData(zeros(n_met, n_samples), (NMRlab.coords(partialSpects,2), NMRlab.coords(processedData,2)))
A = real.(partialSpects.dat)
for i in 1:n_samples
    b=real.(processedData.dat[:,i])
    rawamounts_nnls[:,i] = NonNegLeastSquares.nnls(A, b)
end
rawamounts_nnls 

In [ ]:
Plots.bar(
     NMRlab.coords(rawamounts_nnls,1), 
     rawamounts_nnls[:,2], 
     xticks=(0.5:39, NMRlab.coords(rawamounts_nnls,1)),
     legend=false,
     size=(900,600),
     ylabel="relative amount (a.u.)", title=NMRlab.coords(rawamounts_nnls,2)[1],xrotation=60,)

In [ ]:
n=1
fittedSpectra = SpectData(partialSpects * rawamounts_nnls, (NMRlab.coords(partialSpects,1), NMRlab.coords(rawamounts_tikonov,2)))
Plots.plot(NMRlab.coords(fittedSpectra,1)./600 .+ 4.78, real(fittedSpectra)[:,n], xaxis=:flip, fmt=:svg,label="fitted spectrum",legend=:topleft)
Plots.plot!(NMRlab.coords(processedData,1)./600 .+ 4.78, 0.5e3 .+ real.(processedData[:,n]), xaxis=:flip, fmt=:svg, 
label="measured spectrum",ylims=10e3 .* (-0.1,1.5),xlims=0.0.+(-0.1,8.7))

In [ ]:
Plots.heatmap(rawamounts_nnls,
    xticks=(1:18,NMRlab.coords(rawamounts_nnls,2)),xrotation=60,
    yticks=(1:38,NMRlab.coords(rawamounts_nnls,1)),
    size=(800,800))